In [2]:
# PREPROCESSING DATA CUACA

import pandas as pd  # library untuk manipulasi data berbentuk tabel (DataFrame)
import numpy as np   # library untuk operasi numerik (digunakan untuk NaN, dll)

# LOAD DATA
df = pd.read_excel("Logbook Cuaca.xlsx")
# membaca file excel dan menyimpannya ke dalam DataFrame bernama df

# CEK MISSING SEBELUM
print("Missing BEFORE:\n", df.isnull().sum())
# menampilkan jumlah nilai kosong (NaN) di setiap kolom sebelum preprocessing

# KONVERSI NUMERIK
numeric_cols = ['Suhu (celcius)', 'Kelembapan (%)', 'Angin (km/h)', 'Intensitas UV']
# daftar kolom yang seharusnya bertipe numerik

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # mengubah isi kolom menjadi numerik (float/int)
    # errors='coerce' → jika ada nilai tidak valid (misal string), akan diubah menjadi NaN

# SORT DATA
df = df.sort_values(by=['Lokasi', 'Waktu', 'No']).reset_index(drop=True)
# mengurutkan data berdasarkan:
# 1. Lokasi (agar tiap lokasi terkelompok)
# 2. Waktu (09 dan 15 dipisah)
# 3. No (urutan nomor data dalam tiap grup)
# reset_index(drop=True) → reset index agar berurutan dari 0 tanpa menyimpan index lama

# FUNGSI LAGRANGE ORDE 2
def lagrange_orde_2(x0, x1, x2, y0, y1, y2, x):
    # x0, x1, x2 = nilai kolom No pada titik referensi
    # y0, y1, y2 = nilai fitur pada titik tersebut
    # x = nilai No pada titik yang ingin dicari

    L0 = ((x - x1)*(x - x2)) / ((x0 - x1)*(x0 - x2))
    L1 = ((x - x0)*(x - x2)) / ((x1 - x0)*(x1 - x2))
    L2 = ((x - x0)*(x - x1)) / ((x2 - x0)*(x2 - x1))

    return y0*L0 + y1*L1 + y2*L2

# TAHAP 1: INTERPOLASI LAGRANGE UNTUK SEMUA FITUR NUMERIK
df_result = df.copy()
# membuat salinan data agar data asli tidak berubah

filled_total = 0
skipped_total = 0

for col in numeric_cols:

    filled_count = 0
    skipped_count = 0

    for (lokasi, waktu), group in df.groupby(['Lokasi', 'Waktu']):
        # membagi data menjadi grup berdasarkan kombinasi Lokasi dan Waktu

        group = group.reset_index()
        # index lama disimpan di kolom 'index'

        for i in range(len(group)):

            if pd.isna(group.loc[i, col]):

                valid_points = group[group[col].notna()]

                if len(valid_points) < 3:
                    skipped_count += 1
                    continue

                target_no = group.loc[i, 'No']

                valid_points = valid_points.copy()
                valid_points['distance'] = valid_points['No'].apply(
                    lambda x: abs(x - target_no)
                )
                # menghitung jarak antara No tiap titik valid dengan No target

                nearest = valid_points.nsmallest(3, 'distance')
                # ambil 3 titik dengan jarak No terkecil

                if len(nearest) < 3:
                    skipped_count += 1
                    continue

                x_vals = nearest['No'].values
                y_vals = nearest[col].values

                try:
                    hasil = lagrange_orde_2(
                        x_vals[0], x_vals[1], x_vals[2],
                        y_vals[0], y_vals[1], y_vals[2],
                        target_no
                    )
                    # clipping: hasil tidak boleh kurang dari 0 (tidak logis secara fisik)
                    hasil = max(hasil, 0)
                except:
                    skipped_count += 1
                    continue

                original_index = group.loc[i, 'index']
                df_result.loc[original_index, col] = hasil
                filled_count += 1

    print(f"\nKolom: {col}")
    print("  Berhasil diisi:", filled_count)
    print("  Skip:", skipped_count)

    filled_total += filled_count
    skipped_total += skipped_count

print("\nTOTAL BERHASIL (numerik):", filled_total)
print("TOTAL SKIP (numerik):", skipped_total)

# TAHAP 2: ISI LABEL CUACA YANG KOSONG
# Baris yang Cuaca-nya kosong fiturnya juga semua kosong sebelum interpolasi.
# Setelah fitur terisi via Lagrange, label Cuaca ditentukan dengan cara:
# membandingkan hasil interpolasi fitur baris target dengan nilai fitur
# ketiga titik referensi yang digunakan saat interpolasi.
# Titik referensi dengan total selisih fitur terkecil dianggap paling mirip,
# dan label Cuacanya disalin ke baris target.

cuaca_filled = 0
cuaca_skipped = 0

df_cuaca_missing = df_result[df_result['Cuaca'].isna()].copy()

print(f"\nJumlah baris Cuaca kosong: {len(df_cuaca_missing)}")

for idx in df_cuaca_missing.index:

    # ambil lokasi dan waktu baris target untuk mencari grup yang sama
    lokasi_target = df_result.loc[idx, 'Lokasi']
    waktu_target  = df_result.loc[idx, 'Waktu']
    no_target     = df_result.loc[idx, 'No']

    # ambil seluruh grup yang sama (Lokasi + Waktu) dari data asli
    group = df[
        (df['Lokasi'] == lokasi_target) &
        (df['Waktu']  == waktu_target)
    ].copy()

    # titik referensi = baris dalam grup yang fiturnya lengkap (tidak ada NaN di numeric_cols)
    valid_points = group[group[numeric_cols].notna().all(axis=1)].copy()

    if len(valid_points) < 3:
        cuaca_skipped += 1
        continue

    # ambil 3 titik terdekat berdasarkan No (sama dengan yang dipakai Lagrange)
    valid_points['distance_no'] = valid_points['No'].apply(lambda x: abs(x - no_target))
    nearest = valid_points.nsmallest(3, 'distance_no')

    # ambil nilai fitur baris target setelah diinterpolasi
    target_features = df_result.loc[idx, numeric_cols]

    if target_features.isna().any():
        # jika fitur target masih ada yang kosong, skip
        cuaca_skipped += 1
        continue

    # hitung total selisih absolut antara fitur target dengan fitur tiap titik referensi
    # titik yang total selisihnya paling kecil = paling mirip = label Cuacanya yang dipakai
    min_selisih = float('inf')
    cuaca_hasil = None

    for _, ref_row in nearest.iterrows():
        selisih_total = sum(
            abs(target_features[col] - ref_row[col])
            for col in numeric_cols
        )
        # jumlah selisih absolut semua fitur antara target dan titik referensi ini

        if selisih_total < min_selisih:
            min_selisih  = selisih_total
            cuaca_hasil  = ref_row['Cuaca']
            # simpan Cuaca dari titik referensi yang paling mirip

    if cuaca_hasil is not None:
        df_result.loc[idx, 'Cuaca'] = cuaca_hasil
        cuaca_filled += 1
    else:
        cuaca_skipped += 1

print(f"Cuaca berhasil diisi : {cuaca_filled}")
print(f"Cuaca skip           : {cuaca_skipped}")

# CEK MISSING SESUDAH
print("\nMissing AFTER:\n", df_result.isnull().sum())

# PREVIEW
print("\nPreview Data:\n", df_result.head())

Missing BEFORE:
 No                 0
Tanggal            0
Waktu              0
Lokasi             0
Suhu (celcius)    10
Kelembapan (%)    11
Angin (km/h)      88
Intensitas UV     11
Cuaca             10
dtype: int64

Kolom: Suhu (celcius)
  Berhasil diisi: 10
  Skip: 0

Kolom: Kelembapan (%)
  Berhasil diisi: 11
  Skip: 0

Kolom: Angin (km/h)
  Berhasil diisi: 88
  Skip: 0

Kolom: Intensitas UV
  Berhasil diisi: 11
  Skip: 0

TOTAL BERHASIL (numerik): 120
TOTAL SKIP (numerik): 0

Jumlah baris Cuaca kosong: 10
Cuaca berhasil diisi : 10
Cuaca skip           : 0

Missing AFTER:
 No                0
Tanggal           0
Waktu             0
Lokasi            0
Suhu (celcius)    0
Kelembapan (%)    0
Angin (km/h)      0
Intensitas UV     0
Cuaca             0
dtype: int64

Preview Data:
    No    Tanggal  Waktu         Lokasi  Suhu (celcius)  Kelembapan (%)  \
0   2 2026-02-04    9.0  Jakarta Barat            27.0            81.0   
1  12 2026-02-05    9.0  Jakarta Barat            26.0   

In [3]:
# SIMPAN DATA HASIL PREPROCESSING KE EXCEL

filename = "data_preprocessed.xlsx"
# nama file output

# Urutkan data berdasarkan kolom 'No' yang sudah ada
df_result = df_result.sort_values(by='No').reset_index(drop=True)

df_result.to_excel(filename, index=False)
# menyimpan dataframe hasil preprocessing ke file excel
# index=False → agar index tidak ikut tersimpan sebagai kolom

print(f"Data preprocessing disimpan ke {filename}")

Data preprocessing disimpan ke data_preprocessed.xlsx


In [4]:
# K-FOLD (5 FOLD, BERURUTAN, TANPA SHUFFLE)

import numpy as np  # import numpy untuk operasi numerik

data = df_result.reset_index(drop=True)  # reset index agar urut dari 0 sampai n-1 tanpa loncatan index lama

n_samples = len(data)  # jumlah total data (n), digunakan untuk menentukan pembagian fold
k = 5  # jumlah fold yang diinginkan (5 fold)

# HITUNG UKURAN TIAP FOLD

fold_sizes = [n_samples // k] * k
# n_samples // k = pembagian bulat (floor division)
# contoh: 103 data / 5 fold = 20 (sisa 3)
# maka awalnya semua fold diisi 20 data → [20, 20, 20, 20, 20]

for i in range(n_samples % k):
    # n_samples % k = sisa pembagian
    # contoh: 103 % 5 = 3 → berarti ada 3 data sisa yang belum terbagi

    fold_sizes[i] += 1
    # sisa dibagikan ke fold awal satu per satu
    # hasil akhir contoh: [21, 21, 21, 20, 20]

# MEMBAGI DATA SECARA BERURUTAN

folds = []  # list untuk menyimpan setiap fold
current = 0  # penanda posisi awal slicing data

for fold_size in fold_sizes:

    start = current
    # index awal slice

    end = current + fold_size
    # index akhir slice (tidak termasuk end, sesuai aturan iloc)

    fold = data.iloc[start:end]
    # mengambil subset data dari start sampai end-1 (berurutan, tanpa shuffle)

    folds.append(fold)
    # menyimpan fold ke dalam list

    current = end
    # menggeser pointer ke posisi berikutnya untuk fold selanjutnya

# CEK HASIL PEMBAGIAN

for i, fold in enumerate(folds):
    print(f"Fold {i+1}: {len(fold)} data")
    # menampilkan jumlah data di tiap fold untuk memastikan pembagian benar

Fold 1: 192 data
Fold 2: 192 data
Fold 3: 192 data
Fold 4: 192 data
Fold 5: 192 data


In [5]:
# SIMPAN SETIAP FOLD KE FILE EXCEL TERPISAH

for i, fold in enumerate(folds):

    filename = f"fold_{i+1}.xlsx"
    # membuat nama file: fold_1.xlsx, fold_2.xlsx, ..., fold_5.xlsx

    fold.to_excel(filename, index=False)
    # menyimpan DataFrame ke file excel
    # index=False → agar index tidak ikut tersimpan sebagai kolom

    print(f"Fold {i+1} disimpan ke {filename}")
    # konfirmasi bahwa file berhasil disimpan

Fold 1 disimpan ke fold_1.xlsx
Fold 2 disimpan ke fold_2.xlsx
Fold 3 disimpan ke fold_3.xlsx
Fold 4 disimpan ke fold_4.xlsx
Fold 5 disimpan ke fold_5.xlsx


In [6]:
# TRAINING MODEL RANDOM FOREST

from sklearn.ensemble import RandomForestClassifier  # model Random Forest
from sklearn.preprocessing import LabelEncoder       # untuk encoding data kategorikal

# PREPARE DATA

data = df_result.reset_index(drop=True)  # memastikan index rapi

# ENCODING FITUR KATEGORIKAL

le_lokasi = LabelEncoder()  # encoder untuk kolom Lokasi
data['Lokasi'] = le_lokasi.fit_transform(data['Lokasi'])
# mengubah lokasi (string) menjadi angka

# FITUR WAKTU
# Waktu sudah numerik (09, 15), jadi bisa langsung digunakan
# Jika masih string, pastikan sudah numerik:
data['Waktu'] = pd.to_numeric(data['Waktu'], errors='coerce')

# PILIH FITUR (X) DAN TARGET (y)

X = data[['Suhu (celcius)', 'Kelembapan (%)', 'Angin (km/h)', 'Intensitas UV', 'Lokasi', 'Waktu']]
# fitur yang digunakan:
# - Suhu
# - Kelembapan
# - Angin
# - UV
# - Lokasi (sudah di-encode)
# - Waktu (jam)

y = data['Cuaca']  # target

# ENCODING TARGET

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
# mengubah label cuaca menjadi numerik

# TRAIN MODEL FULL DATA

model_full = RandomForestClassifier(
    n_estimators=1000,
    criterion='gini',
    max_features='sqrt',
    bootstrap=True,
    max_depth=None,
    max_leaf_nodes=None,
    ccp_alpha=0.0,
    min_samples_split=2,
    min_samples_leaf=1,
)
model_full.fit(X, y_encoded)
# melatih model dengan seluruh data

# TRAIN MODEL PER FOLD

models_fold = []  # menyimpan model tiap fold

for i in range(len(folds)):

    # ambil semua fold kecuali fold ke-i sebagai training
    train_folds = [folds[j] for j in range(len(folds)) if j != i]

    train_data = pd.concat(train_folds).reset_index(drop=True)

    # encoding lokasi di tiap fold (harus konsisten)
    train_data['Lokasi'] = le_lokasi.transform(train_data['Lokasi'])

    # pastikan waktu numerik
    train_data['Waktu'] = pd.to_numeric(train_data['Waktu'], errors='coerce')

    # split fitur dan target
    X_train = train_data[['Suhu (celcius)', 'Kelembapan (%)', 'Angin (km/h)', 'Intensitas UV', 'Lokasi', 'Waktu']]
    y_train = train_data['Cuaca']

    # encoding target
    y_train_encoded = le_target.transform(y_train)

    # train model
    model = RandomForestClassifier(
        n_estimators=1000,
        criterion='gini',
        max_features='sqrt',
        bootstrap=True,
        max_depth=None,
        max_leaf_nodes=None,
        ccp_alpha=0.0,
        min_samples_split=2,
        min_samples_leaf=1
    )
    model.fit(X_train, y_train_encoded)

    models_fold.append(model)


print("Jumlah model fold:", len(models_fold))

Jumlah model fold: 5


In [7]:
# SIMPAN SEMUA MODEL KE FILE

import joblib  # library untuk menyimpan model machine learning

# SIMPAN MODEL FULL DATA
joblib.dump(model_full, "model_full.pkl")

# SIMPAN MODEL PER FOLD
for i, model in enumerate(models_fold):

    filename = f"model_fold_{i+1}.pkl"
    # nama file: model_fold_1.pkl sampai model_fold_5.pkl

    joblib.dump(model, filename)
    # menyimpan model ke file